## Preprocessing 점포-행정동


data load

In [1]:
import pandas as pd
import os

# File path
file_path = '../../data/raw/서울시 상권분석서비스(점포-행정동).csv'

# Load data
try:
    df = pd.read_csv(file_path, encoding='cp949')
    print("Data loaded successfully.")
    display(df.head())
except Exception as e:
    print(f"Error loading data: {e}")

Data loaded successfully.


,기준_년분기_코드,행정동_코드,행정동_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수
0,20254,11740700,둔촌2동,CS300043,전자상거래업,35,35,0,0,0,0,0
1,20254,11740700,둔촌2동,CS300042,주유소,7,7,29,2,0,0,0
2,20254,11740700,둔촌2동,CS300041,예술품,3,3,0,0,0,0,0
3,20254,11740700,둔촌2동,CS300040,재생용품 판매점,3,3,0,0,0,0,0
4,20254,11740700,둔촌2동,CS300039,모터사이클및부품,2,2,0,0,0,0,0


category mapping dictionary

In [2]:
category_map = {
    # 외식/F&B
    '한식음식점': '외식/F&B', '중식음식점': '외식/F&B', '일식음식점': '외식/F&B',
    '양식음식점': '외식/F&B', '제과점': '외식/F&B', '패스트푸드점': '외식/F&B',
    '치킨전문점': '외식/F&B', '분식전문점': '외식/F&B', '호프-간이주점': '외식/F&B',
    '커피-음료': '외식/F&B',
    # 교육
    '일반교습학원': '교육', '외국어학원': '교육', '예술학원': '교육',
    '컴퓨터학원': '교육', '스포츠 강습': '교육',  # 띄어쓰기 주의
    # 의료/건강
    '일반의원': '의료/건강', '치과의원': '의료/건강', '한의원': '의료/건강',
    '동물병원': '의료/건강', '의약품': '의료/건강', '의료기기': '의료/건강',
    # 전문서비스
    '변호사사무소': '전문서비스', '변리사사무소': '전문서비스', '법무사사무소': '전문서비스',
    '기타법무서비스': '전문서비스', '회계사사무소': '전문서비스', '세무사사무소': '전문서비스',
    # 오락/여가
    '당구장': '오락/여가', '골프연습장': '오락/여가', '볼링장': '오락/여가',
    'PC방': '오락/여가', '전자게임장': '오락/여가', '기타오락장': '오락/여가',
    '복권방': '오락/여가', '노래방': '오락/여가', '독서실': '오락/여가',
    'DVD방': '오락/여가', '비디오/서적임대': '오락/여가',
    # 생활서비스
    '미용실': '생활서비스', '네일숍': '생활서비스', '피부관리실': '생활서비스',
    '세탁소': '생활서비스', '스포츠클럽': '생활서비스', '자동차수리': '생활서비스',
    '자동차미용': '생활서비스', '모터사이클수리': '생활서비스', '통신기기수리': '생활서비스',
    '가전제품수리': '생활서비스', '사진관': '생활서비스', '통번역서비스': '생활서비스',
    '건축물청소': '생활서비스', '녹음실': '생활서비스', '여행사': '생활서비스',
    # 숙박/부동산
    '여관': '숙박/부동산', '게스트하우스': '숙박/부동산',
    '고시원': '숙박/부동산', '부동산중개업': '숙박/부동산',
    # 식품/식재료
    '슈퍼마켓': '식품/식재료', '편의점': '식품/식재료', '주류도매': '식품/식재료',
    '미곡판매': '식품/식재료', '육류판매': '식품/식재료', '수산물판매': '식품/식재료',
    '청과상': '식품/식재료', '반찬가게': '식품/식재료',
    # 패션/뷰티
    '일반의류': '패션/뷰티', '한복점': '패션/뷰티', '유아의류': '패션/뷰티',
    '신발': '패션/뷰티', '가방': '패션/뷰티', '안경': '패션/뷰티',
    '시계및귀금속': '패션/뷰티', '화장품': '패션/뷰티', '미용재료': '패션/뷰티',
    '의류임대': '패션/뷰티',
    # 전자/디지털
    '핸드폰': '전자/디지털', '컴퓨터및주변장치판매': '전자/디지털',
    '가전제품': '전자/디지털', '전자상거래업': '전자/디지털',
    # 생활/홈인테리어
    '가구': '생활/홈인테리어', '중고가구': '생활/홈인테리어', '철물점': '생활/홈인테리어',
    '인테리어': '생활/홈인테리어', '조명용품': '생활/홈인테리어', '화초': '생활/홈인테리어',
    '섬유제품': '생활/홈인테리어', '가정용품임대': '생활/홈인테리어',
    # 자동차/이동수단
    '중고차판매': '자동차/이동수단', '자동차부품': '자동차/이동수단',
    '모터사이클및부품': '자동차/이동수단', '주유소': '자동차/이동수단',
    '자전거 및 기타운송장비': '자동차/이동수단',
}

기타소매 드랍 리스트

In [3]:
drop_list = ['서적', '문구', '운동/경기용품', '완구', '악기', '애완동물', '예술품', '재생용품 판매점']

**검증** 

In [4]:
# 실제 데이터 업종 목록
actual_list = set(df['서비스_업종_코드_명'].unique())
map_list = set(category_map.keys())
drop_set = set(drop_list)
expected_list = map_list | drop_set

print("=" * 40)
print("✅ 실제 데이터에 있는데 매핑/드랍에 없는 것 (누락)")
print(actual_list - expected_list)

print("\n⚠️ 매핑/드랍에 있는데 실제 데이터에 없는 것 (오타 의심)")
print(expected_list - actual_list)

print("\n📊 실제 업종 수:", len(actual_list))
print("📊 매핑 업종 수:", len(map_list))
print("📊 드랍 업종 수:", len(drop_set))
print("=" * 40)

✅ 실제 데이터에 있는데 매핑/드랍에 없는 것 (누락)
set()

⚠️ 매핑/드랍에 있는데 실제 데이터에 없는 것 (오타 의심)
set()

📊 실제 업종 수: 100
📊 매핑 업종 수: 92
📊 드랍 업종 수: 8


**전처리**

**1. 카테고리 전처리** 

기타소매 drop

In [5]:
df = df[~df['서비스_업종_코드_명'].isin(drop_list)]

카테고리 컬럼 추가

In [6]:
df['카테고리'] = df['서비스_업종_코드_명'].map(category_map)

혹시 매핑 안 된 행 최종 확인

In [7]:
unmapped = df[df['카테고리'].isna()]['서비스_업종_코드_명'].unique()
print(f"\n❌ 매핑 안 된 업종 (있으면 category_map 추가 필요): {unmapped}")


❌ 매핑 안 된 업종 (있으면 category_map 추가 필요): []


In [8]:
print(f"\n✅ 전처리 완료: {df.shape}")
df.head()


✅ 전처리 완료: (129835, 13)


,기준_년분기_코드,행정동_코드,행정동_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수,카테고리
0,20254,11740700,둔촌2동,CS300043,전자상거래업,35,35,0,0,0,0,0,전자/디지털
1,20254,11740700,둔촌2동,CS300042,주유소,7,7,29,2,0,0,0,자동차/이동수단
4,20254,11740700,둔촌2동,CS300039,모터사이클및부품,2,2,0,0,0,0,0,자동차/이동수단
5,20254,11740700,둔촌2동,CS300038,자동차부품,4,5,0,0,0,0,1,자동차/이동수단
6,20254,11740700,둔촌2동,CS300037,중고차판매,1,1,0,0,0,0,0,자동차/이동수단


마지막 테스트

In [9]:
# 카테고리별 업종 확인
print('=== 카테고리별 매핑된 업종 목록 ===')
for cat in sorted(df['카테고리'].unique()):
    업종들 = sorted(df[df['카테고리'] == cat]['서비스_업종_코드_명'].unique())
    print(f'\n[{cat}] {len(업종들)}개')
    print(업종들)

# NaN 체크
print('\n=== 매핑 안 된 행 수 ===')
print(df['카테고리'].isna().sum())

# 카테고리 분포
print('\n=== 카테고리별 행 수 ===')
print(df['카테고리'].value_counts())

=== 카테고리별 매핑된 업종 목록 ===

[교육] 5개
['스포츠 강습', '예술학원', '외국어학원', '일반교습학원', '컴퓨터학원']

[생활/홈인테리어] 8개
['가구', '가정용품임대', '섬유제품', '인테리어', '조명용품', '중고가구', '철물점', '화초']

[생활서비스] 15개
['가전제품수리', '건축물청소', '네일숍', '녹음실', '모터사이클수리', '미용실', '사진관', '세탁소', '스포츠클럽', '여행사', '자동차미용', '자동차수리', '통번역서비스', '통신기기수리', '피부관리실']

[숙박/부동산] 4개
['게스트하우스', '고시원', '부동산중개업', '여관']

[식품/식재료] 8개
['미곡판매', '반찬가게', '수산물판매', '슈퍼마켓', '육류판매', '주류도매', '청과상', '편의점']

[오락/여가] 11개
['DVD방', 'PC방', '골프연습장', '기타오락장', '노래방', '당구장', '독서실', '복권방', '볼링장', '비디오/서적임대', '전자게임장']

[외식/F&B] 10개
['분식전문점', '양식음식점', '일식음식점', '제과점', '중식음식점', '치킨전문점', '커피-음료', '패스트푸드점', '한식음식점', '호프-간이주점']

[의료/건강] 6개
['동물병원', '의료기기', '의약품', '일반의원', '치과의원', '한의원']

[자동차/이동수단] 5개
['모터사이클및부품', '자동차부품', '자전거 및 기타운송장비', '주유소', '중고차판매']

[전문서비스] 6개
['기타법무서비스', '법무사사무소', '변리사사무소', '변호사사무소', '세무사사무소', '회계사사무소']

[전자/디지털] 4개
['가전제품', '전자상거래업', '컴퓨터및주변장치판매', '핸드폰']

[패션/뷰티] 10개
['가방', '미용재료', '시계및귀금속', '신발', '안경', '유아의류', '의류임대', '일반의류', '한복점', '화장품']

=== 매핑 안 된 행 수 ===
0

==

**2. "구" 컬럼 만들기**

In [10]:
df['자치구_코드'] = df['행정동_코드'].astype(str).str[0:5].astype(int)

In [11]:
gu_map = {
    11110: '종로구', 11140: '중구', 11170: '용산구',
    11200: '성동구', 11215: '광진구', 11230: '동대문구',
    11260: '중랑구', 11290: '성북구', 11305: '강북구',
    11320: '도봉구', 11350: '노원구', 11380: '은평구',
    11410: '서대문구', 11440: '마포구', 11470: '양천구',
    11500: '강서구', 11530: '구로구', 11545: '금천구',
    11560: '영등포구', 11590: '동작구', 11620: '관악구',
    11650: '서초구', 11680: '강남구', 11710: '송파구',
    11740: '강동구'
}

df['자치구_코드_명'] = df['자치구_코드'].map(gu_map)


검증

In [12]:
print(df[['자치구_코드', '자치구_코드_명']].drop_duplicates().sort_values('자치구_코드'))
print('\n매핑 안 된 것:', df['자치구_코드_명'].isna().sum()) 

       자치구_코드 자치구_코드_명
34048   11110      종로구
32793   11140       중구
31512   11170      용산구
30130   11200      성동구
28799   11215      광진구
27613   11230     동대문구
26284   11260      중랑구
24694   11290      성북구
23632   11305      강북구
22513   11320      도봉구
21007   11350      노원구
19660   11380      은평구
18531   11410     서대문구
17108   11440      마포구
15644   11470      양천구
13932   11500      강서구
12570   11530      구로구
11739   11545      금천구
10188   11560     영등포구
8930    11590      동작구
7180    11620      관악구
5593    11650      서초구
3708    11680      강남구
1490    11710      송파구
0       11740      강동구

매핑 안 된 것: 0


In [13]:
df.head()

,기준_년분기_코드,행정동_코드,행정동_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수,카테고리,자치구_코드,자치구_코드_명
0,20254,11740700,둔촌2동,CS300043,전자상거래업,35,35,0,0,0,0,0,전자/디지털,11740,강동구
1,20254,11740700,둔촌2동,CS300042,주유소,7,7,29,2,0,0,0,자동차/이동수단,11740,강동구
4,20254,11740700,둔촌2동,CS300039,모터사이클및부품,2,2,0,0,0,0,0,자동차/이동수단,11740,강동구
5,20254,11740700,둔촌2동,CS300038,자동차부품,4,5,0,0,0,0,1,자동차/이동수단,11740,강동구
6,20254,11740700,둔촌2동,CS300037,중고차판매,1,1,0,0,0,0,0,자동차/이동수단,11740,강동구


**3. 각 행정동의 전체 점포 수 만들기**

In [14]:
dong_total = df.groupby('행정동_코드')['점포_수'].sum().reset_index()
dong_total.rename(columns={'점포_수': '동_총_점포_수'}, inplace=True)
df = df.merge(dong_total, on='행정동_코드', how='left')

검증

In [15]:
print(df[['행정동_코드_명', '동_총_점포_수']].drop_duplicates().sort_values('동_총_점포_수', ascending=False).head(10))
print('\n매핑 안 된 것:', df['동_총_점포_수'].isna().sum()) 

          행정동_코드_명  동_총_점포_수
4195          역삼1동     43328
30661          신당동     41002
31835  종로1?2?3?4가동     37016
16279          서교동     31169
11456          가산동     30773
6324          서초3동     26128
1925          문정2동     22123
10526          여의동     20074
30825          광희동     18980
29460         한강로동     18829

매핑 안 된 것: 0


**4. 4분기 데이터만 남기기**

In [16]:
df = df[df['기준_년분기_코드'] == 20254]

**마무리**

In [17]:
df.head()

,기준_년분기_코드,행정동_코드,행정동_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수,카테고리,자치구_코드,자치구_코드_명,동_총_점포_수
0,20254,11740700,둔촌2동,CS300043,전자상거래업,35,35,0,0,0,0,0,전자/디지털,11740,강동구,4114
1,20254,11740700,둔촌2동,CS300042,주유소,7,7,29,2,0,0,0,자동차/이동수단,11740,강동구,4114
2,20254,11740700,둔촌2동,CS300039,모터사이클및부품,2,2,0,0,0,0,0,자동차/이동수단,11740,강동구,4114
3,20254,11740700,둔촌2동,CS300038,자동차부품,4,5,0,0,0,0,1,자동차/이동수단,11740,강동구,4114
4,20254,11740700,둔촌2동,CS300037,중고차판매,1,1,0,0,0,0,0,자동차/이동수단,11740,강동구,4114


In [18]:
df = df[['기준_년분기_코드', '자치구_코드', '자치구_코드_명', '행정동_코드', '행정동_코드_명', 
         '서비스_업종_코드', '서비스_업종_코드_명', '카테고리', '점포_수', '동_총_점포_수',
         '유사_업종_점포_수', '개업_율', '개업_점포_수', '폐업_률', '폐업_점포_수', '프랜차이즈_점포_수']]

df.head()

,기준_년분기_코드,자치구_코드,자치구_코드_명,행정동_코드,행정동_코드_명,서비스_업종_코드,서비스_업종_코드_명,카테고리,점포_수,동_총_점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수
0,20254,11740,강동구,11740700,둔촌2동,CS300043,전자상거래업,전자/디지털,35,4114,35,0,0,0,0,0
1,20254,11740,강동구,11740700,둔촌2동,CS300042,주유소,자동차/이동수단,7,4114,7,29,2,0,0,0
2,20254,11740,강동구,11740700,둔촌2동,CS300039,모터사이클및부품,자동차/이동수단,2,4114,2,0,0,0,0,0
3,20254,11740,강동구,11740700,둔촌2동,CS300038,자동차부품,자동차/이동수단,4,4114,5,0,0,0,0,1
4,20254,11740,강동구,11740700,둔촌2동,CS300037,중고차판매,자동차/이동수단,1,4114,1,0,0,0,0,0


**5. 카테고리별로 합산**

In [ ]:
업종_cols = [c for c in df.columns if '업종' in c]
df_drop = df.drop(columns=업종_cols)

# 코드성 컬럼은 group_key에 추가
code_cols = ['기준_년분기_코드', '자치구_코드', '행정동_코드']
str_cols = df_drop.select_dtypes(include='object').columns.tolist()
group_keys = str_cols + code_cols

# 비율 컬럼 제외하고 sum
rate_cols = ['개업_율', '폐업_률']
num_cols = [
    c for c in df_drop.select_dtypes(include='number').columns
    if c not in rate_cols and c not in code_cols
]

agg_dict = {c: 'sum' for c in num_cols}
agg_dict['동_총_점포_수'] = 'first'

df_grouped = (
    df_drop
    .drop(columns=rate_cols)
    .groupby(group_keys, as_index=False)
    .agg(agg_dict)
)

df_grouped['개업_율'] = df_grouped['개업_점포_수'] / df_grouped['점포_수']
df_grouped['폐업_률'] = df_grouped['폐업_점포_수'] / df_grouped['점포_수']
df_grouped[['개업_율', '폐업_률']] = df_grouped[['개업_율', '폐업_률']].fillna(0)

print(df_grouped.shape)
df_grouped.head()

(5067, 13)


,자치구_코드_명,행정동_코드_명,카테고리,기준_년분기_코드,자치구_코드,행정동_코드,점포_수,동_총_점포_수,개업_점포_수,폐업_점포_수,프랜차이즈_점포_수,개업_율,폐업_률
0,강남구,개포1동,교육,81016,46720,46722640,29,736,0,0,2,0.0,0.000000
1,강남구,개포1동,생활/홈인테리어,81016,46720,46722640,11,736,0,2,0,0.0,0.181818
2,강남구,개포1동,생활서비스,222794,128480,128487260,29,736,0,0,1,0.0,0.000000
3,강남구,개포1동,숙박/부동산,20254,11680,11680660,40,736,0,1,0,0.0,0.025000
4,강남구,개포1동,식품/식재료,121524,70080,70083960,13,736,0,0,4,0.0,0.000000


**저장**

In [20]:
output_path = '../../data/preprocessed/상권분석_점포_행정동_전처리.csv'
df_grouped.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✅ 최종 데이터가 저장되었습니다: {output_path}")

✅ 최종 데이터가 저장되었습니다: ../../data/preprocessed/상권분석_점포_행정동_전처리.csv
